# Tuned BCE-SASRec for MOOCCubeX: three-seed Colab experiment

This notebook performs validation-only hyperparameter tuning and then trains the proposed Behaviour-and-Concept-Enhanced SASRec model with seeds **42, 2026, and 3407**. It uses chronological train/validation/test splits, keeps the test set isolated until final evaluation, uses stable FP32 training, saves every checkpoint to Google Drive, and produces complete metric tables, static figures, and an animated learning curve.

**Research protocol:** 15 Optuna trials, up to 8 epochs per trial, followed by up to 30 epochs for each final seed with early stopping. An NDCG target is reported but never forced.


## 1. Mount Drive and install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, subprocess, importlib.util
packages=[]
for module,package in [('optuna','optuna'),('pyarrow','pyarrow'),('PIL','pillow')]:
    if importlib.util.find_spec(module) is None:packages.append(package)
if packages:subprocess.check_call([sys.executable,'-m','pip','install','-q',*packages])
print('Dependencies ready.')


## 2. Restore the processed dataset when necessary

In [ ]:
from pathlib import Path
import shutil

DATA_ROOT=Path('/content/drive/MyDrive/DataCon')
PROCESSED_EXPECTED=DATA_ROOT/'processed'
required=[PROCESSED_EXPECTED/'splits/train.parquet',PROCESSED_EXPECTED/'splits/valid.parquet',
          PROCESSED_EXPECTED/'splits/test.parquet',PROCESSED_EXPECTED/'graph/video_metadata.parquet',
          PROCESSED_EXPECTED/'graph/concept_video_edges.parquet']

if not all(p.exists() for p in required):
    zip_candidates=list(DATA_ROOT.rglob('MOOCCubeX_processed.zip'))
    if not zip_candidates:
        raise FileNotFoundError('Upload MOOCCubeX_processed.zip to /content/drive/MyDrive/DataCon.')
    print('Extracting:',zip_candidates[0])
    shutil.unpack_archive(str(zip_candidates[0]),str(DATA_ROOT))

missing=[str(p) for p in required if not p.exists()]
if missing:raise FileNotFoundError(f'Missing processed files: {missing}')
print('Processed dataset ready:',PROCESSED_EXPECTED)


## 3. Verify the GPU

In [ ]:
import torch
print('PyTorch:',torch.__version__)
print('CUDA available:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then reconnect and run again.')
print('GPU:',torch.cuda.get_device_name(0))


## 4. Imports, configuration, and persistent paths

In [ ]:
"""Three-seed validation-tuned BCE-SASRec experiment for Kaggle.

Requires preprocessed MOOCCubeX splits and graph Parquet files in Google Drive.
Hyperparameters are chosen only by validation NDCG@10. The test set is evaluated
after selection for seeds 42, 2026 and 3407. A target is reported, never forced.
"""
import warnings
warnings.filterwarnings("ignore", message="enable_nested_tensor is True")
try:
    from IPython.display import display
except ImportError:
    def display(x): print(x)


## 5. Load and index train, validation, and test data

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
import copy, gc, json, math, os, random, time
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

@dataclass
class Config:
    root: str = '/content/drive/MyDrive/DataCon'
    seed: int = 42
    max_len: int = 50
    max_concepts_per_video: int = 12
    hidden_dim: int = 128
    transformer_layers: int = 2
    attention_heads: int = 4
    feedforward_dim: int = 512
    dropout: float = 0.10
    time_buckets: int = 32
    negatives: int = 50
    batch_size: int = 128
    eval_batch_size: int = 128
    max_epochs: int = 25
    minimum_epochs: int = 10
    early_stopping_patience: int = 5
    early_stopping_min_delta: float = 1e-4
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    concept_loss_weight: float = 0.20
    completion_loss_weight: float = 0.10
    gradient_clip: float = 5.0
    use_mixed_precision: bool = False
    ks: tuple = (5, 10, 20)
    num_workers: int = 2

CFG=Config()

# Prefer preprocessing outputs created in this Kaggle session.
working_processed=Path('/content/drive/MyDrive/DataCon/processed')
search_roots=[working_processed]
matches=[]
for search_root in search_roots:
    if search_root.name=='processed':
        candidates=[search_root/'splits/train.parquet']
    else:
        candidates=[p for p in search_root.rglob('train.parquet') if p.parent.name=='splits']
    for train_path in candidates:
        processed=train_path.parent.parent
        graph=processed/'graph'
        required_graph=['video_metadata.parquet','concept_video_edges.parquet','video_index.parquet','course_video_edges.parquet']
        if train_path.exists() and all((graph/name).exists() for name in required_graph) and (train_path.parent/'valid.parquet').exists() and (train_path.parent/'test.parquet').exists():
            matches.append(processed)
    if matches:break
if not matches:
    raise FileNotFoundError('Could not find complete processed data under /content/drive/MyDrive/DataCon/processed. Run the restore cell first.')
PROCESSED=matches[0];SPLITS=PROCESSED/'splits';GRAPH=PROCESSED/'graph';ROOT=PROCESSED.parent

# All generated artifacts persist in Google Drive.
OUT=Path('/content/drive/MyDrive/DataCon/bce_sasrec_hpo_3seeds');CHECKPOINTS=OUT/'checkpoints';REPORTS=OUT/'reports';EXPLANATIONS=OUT/'explanations'
for p in [OUT,CHECKPOINTS,REPORTS,EXPLANATIONS]:p.mkdir(parents=True,exist_ok=True)
print('Detected processed data:',PROCESSED)
print('Output directory:',OUT)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

seed_everything(CFG.seed)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
print(json.dumps(asdict(CFG),indent=2))


## 6. Audit chronology and leakage

In [ ]:
required=[SPLITS/'train.parquet',SPLITS/'valid.parquet',SPLITS/'test.parquet',
          GRAPH/'video_metadata.parquet',GRAPH/'concept_video_edges.parquet',
          GRAPH/'video_index.parquet',GRAPH/'course_video_edges.parquet']
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError(f'Missing preprocessing outputs: {missing}')

train_df=pd.read_parquet(SPLITS/'train.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)
valid_df=pd.read_parquet(SPLITS/'valid.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)
test_df=pd.read_parquet(SPLITS/'test.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)

required_columns={'user_id','video_id','timestamp','duration_seconds','watched_seconds',
                  'playback_seconds','completion_ratio','segment_count','engagement_weight'}
for name,frame in [('train',train_df),('valid',valid_df),('test',test_df)]:
    absent=required_columns-set(frame.columns)
    if absent: raise ValueError(f'{name} is missing columns: {sorted(absent)}')
    frame['user_id']=frame.user_id.astype(str); frame['video_id']=frame.video_id.astype(str)
    frame['timestamp']=pd.to_numeric(frame.timestamp,errors='coerce').fillna(0).astype('int64')

item_ids=sorted(train_df.video_id.unique()); user_ids=sorted(set(train_df.user_id)|set(valid_df.user_id)|set(test_df.user_id))
item2idx={v:i+1 for i,v in enumerate(item_ids)}; idx2item={i:v for v,i in item2idx.items()}
user2idx={u:i for i,u in enumerate(user_ids)}; idx2user={i:u for u,i in user2idx.items()}
num_items,num_users=len(item2idx),len(user2idx)

def map_frame(frame):
    x=frame[frame.video_id.isin(item2idx)].copy().reset_index(drop=True)
    x['u']=x.user_id.map(user2idx).astype('int64'); x['i']=x.video_id.map(item2idx).astype('int64')
    return x

train=map_frame(train_df);valid=map_frame(valid_df);test=map_frame(test_df)
print({'train':len(train),'validation':len(valid),'test':len(test),'users':num_users,'videos':num_items})
display(pd.DataFrame([
    ['train',len(train),train.u.nunique(),train.i.nunique()],
    ['validation',len(valid),valid.u.nunique(),valid.i.nunique()],
    ['test',len(test),test.u.nunique(),test.i.nunique()]],
    columns=['split','interactions','users','videos']))


## 7. Normalize behavioural features using training data

In [ ]:
train_max=train.groupby('u').timestamp.max()
valid_by_user=valid.set_index('u'); test_by_user=test.set_index('u')
common_users=sorted(set(train_max.index)&set(valid_by_user.index)&set(test_by_user.index))

checks={
 'train_before_or_at_validation':bool((train_max.loc[common_users].values<=valid_by_user.loc[common_users].timestamp.values).all()),
 'validation_before_or_at_test':bool((valid_by_user.loc[common_users].timestamp.values<=test_by_user.loc[common_users].timestamp.values).all()),
 'validation_items_in_training_catalog':bool(valid.i.isin(set(train.i)).all()),
 'test_items_in_training_catalog':bool(test.i.isin(set(train.i)).all()),
 'same_validation_and_test_users':set(valid.u)==set(test.u),
}
display(pd.DataFrame(checks.items(),columns=['check','passed']))
if not all(checks.values()): raise AssertionError('Leakage/chronology audit failed.')


## 8. Build video, concept, course, caption, and metadata tensors

In [ ]:
BEHAVIOUR_COLUMNS=['watched_seconds','playback_seconds','duration_seconds',
                   'completion_ratio','segment_count','engagement_weight']
LOG_BEHAVIOUR={'watched_seconds','playback_seconds','duration_seconds','segment_count'}

def raw_behaviour(frame):
    x=frame[BEHAVIOUR_COLUMNS].astype('float32').replace([np.inf,-np.inf],np.nan).fillna(0).copy()
    for c in LOG_BEHAVIOUR:x[c]=np.log1p(x[c].clip(lower=0))
    return x

train_beh_raw=raw_behaviour(train);beh_mean=train_beh_raw.mean();beh_std=train_beh_raw.std().replace(0,1).fillna(1)
def normalized_behaviour(frame):return ((raw_behaviour(frame)-beh_mean)/beh_std).astype('float32').to_numpy()
train_beh=normalized_behaviour(train);valid_beh=normalized_behaviour(valid);test_beh=normalized_behaviour(test)
normalization={'columns':BEHAVIOUR_COLUMNS,'mean':beh_mean.to_dict(),'std':beh_std.to_dict(),'log1p_columns':sorted(LOG_BEHAVIOUR)}
json.dump(normalization,open(REPORTS/'behaviour_normalization.json','w'),indent=2)
display(pd.DataFrame({'mean':beh_mean,'std':beh_std}))


## 9. Construct chronological histories

In [ ]:
video_index=pd.read_parquet(GRAPH/'video_index.parquet');video_index['video_id']=video_index.video_id.astype(str);video_index['ccid']=video_index.ccid.astype(str)
video_to_ccid=dict(zip(video_index.video_id,video_index.ccid));ccid_to_item={video_to_ccid[v]:item2idx[v] for v in item2idx if v in video_to_ccid}

cv=pd.read_parquet(GRAPH/'concept_video_edges.parquet');cv['concept_id']=cv.concept_id.astype(str);cv['ccid']=cv.ccid.astype(str)
cv=cv[cv.ccid.isin(ccid_to_item)].drop_duplicates(['ccid','concept_id'])
concept_ids=sorted(cv.concept_id.unique());concept2idx={c:i+1 for i,c in enumerate(concept_ids)};idx2concept={i:c for c,i in concept2idx.items()}
item_concepts=np.zeros((num_items+1,CFG.max_concepts_per_video),dtype=np.int64)
for ccid,g in cv.groupby('ccid'):
    ids=[concept2idx[c] for c in g.concept_id.iloc[:CFG.max_concepts_per_video]]
    item_concepts[ccid_to_item[ccid],:len(ids)]=ids

course=pd.read_parquet(GRAPH/'course_video_edges.parquet')
if 'video_id' not in course.columns:
    if 'course_video_id' in course.columns:course=course.rename(columns={'course_video_id':'video_id'})
    else:raise ValueError('course_video_edges.parquet needs video_id or course_video_id')
course['video_id']=course.video_id.astype(str);course['course_id']=course.course_id.astype(str)
course=course[course.video_id.isin(item2idx)].drop_duplicates('video_id')
course_ids=sorted(course.course_id.unique());course2idx={c:i+1 for i,c in enumerate(course_ids)}
item_course=np.zeros(num_items+1,dtype=np.int64)
for row in course.itertuples():item_course[item2idx[row.video_id]]=course2idx[row.course_id]

metadata=pd.read_parquet(GRAPH/'video_metadata.parquet');metadata['video_id']=metadata.video_id.astype(str)
if 'caption_text' in metadata.columns:
    metadata['caption_characters']=metadata.caption_text.fillna('').astype(str).str.len()
if 'ccid' in metadata.columns:
    metadata['concept_count']=metadata.ccid.astype(str).map(cv.groupby('ccid').size()).fillna(0)
META_COLUMNS=['duration_seconds','caption_segment_count','caption_characters','concept_count']
for c in META_COLUMNS:
    if c not in metadata:metadata[c]=0
metadata=metadata.drop_duplicates('video_id').set_index('video_id')
meta_table=pd.DataFrame(index=item_ids,columns=META_COLUMNS,dtype='float32')
for c in META_COLUMNS:meta_table[c]=pd.to_numeric(metadata.reindex(item_ids)[c],errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0).astype('float32')
for c in META_COLUMNS:meta_table[c]=np.log1p(meta_table[c].clip(lower=0))
meta_mean=meta_table.mean();meta_std=meta_table.std().replace(0,1).fillna(1)
meta_table=((meta_table-meta_mean)/meta_std).astype('float32')
item_metadata=np.zeros((num_items+1,len(META_COLUMNS)),dtype=np.float32);item_metadata[1:]=meta_table.to_numpy()

print({'concepts':len(concept_ids),'concept_video_edges':len(cv),'courses':len(course_ids),
       'videos_with_concepts':int((item_concepts!=0).any(1).sum()),'videos_with_courses':int((item_course!=0).sum())})
json.dump({'meta_columns':META_COLUMNS,'mean':meta_mean.to_dict(),'std':meta_std.to_dict()},open(REPORTS/'metadata_normalization.json','w'),indent=2)


## 10. Training examples and data loader

In [ ]:
def build_history(frame,features):
    out={}
    for u,idx in frame.groupby('u',sort=False).groups.items():
        rows=np.asarray(list(idx));g=frame.loc[rows]
        out[int(u)]={'items':g.i.astype(int).tolist(),'times':g.timestamp.astype('int64').tolist(),
                     'behaviour':features[rows].tolist(),'completion':g.completion_ratio.astype(float).tolist()}
    return out

train_h=build_history(train,train_beh);valid_events=build_history(valid,valid_beh);test_events=build_history(test,test_beh)
eval_users=sorted(set(train_h)&set(valid_events)&set(test_events))
valid_hist={u:copy.deepcopy(train_h[u]) for u in eval_users}
test_hist={}
valid_target={u:valid_events[u]['items'][0] for u in eval_users};test_target={u:test_events[u]['items'][0] for u in eval_users}
valid_completion={u:valid_events[u]['completion'][0] for u in eval_users};test_completion={u:test_events[u]['completion'][0] for u in eval_users}
for u in eval_users:
    h=copy.deepcopy(train_h[u])
    for key in ['items','times','behaviour','completion']:h[key].append(valid_events[u][key][0])
    test_hist[u]=h

all_positive={u:set(train_h[u]['items'])|{valid_target[u],test_target[u]} for u in eval_users}
for u in train_h:
    all_positive.setdefault(u,set(train_h[u]['items']))
assert all(valid_target[u] not in valid_hist[u]['items'] for u in eval_users)
assert all(test_target[u] not in test_hist[u]['items'] for u in eval_users)
print('Leakage-free evaluation users:',len(eval_users))


## 11. Proposed BCE-SASRec architecture

In [ ]:
def left_pad(seq,n,pad):
    # Right padding prevents fully masked attention rows under a causal mask.
    seq=list(seq)[-n:];return seq+[pad]*(n-len(seq))

class PrefixDataset(Dataset):
    def __init__(self,histories,max_len):
        self.h=histories;self.max_len=max_len
        self.examples=[(u,t) for u,h in histories.items() for t in range(1,len(h['items']))]
    def __len__(self):return len(self.examples)
    def __getitem__(self,index):
        u,t=self.examples[index];h=self.h[u]
        return (torch.tensor(u),torch.tensor(left_pad(h['items'][:t],self.max_len,0)),
                torch.tensor(left_pad(h['times'][:t],self.max_len,0)),
                torch.tensor(left_pad(h['behaviour'][:t],self.max_len,[0.0]*len(BEHAVIOUR_COLUMNS)),dtype=torch.float32),
                torch.tensor(h['items'][t]),torch.tensor(h['completion'][t],dtype=torch.float32))

train_dataset=PrefixDataset(train_h,CFG.max_len)
train_loader=DataLoader(train_dataset,batch_size=CFG.batch_size,shuffle=True,num_workers=CFG.num_workers,
                        pin_memory=True,persistent_workers=CFG.num_workers>0)
print('Training prefix examples:',len(train_dataset),'batches per epoch:',len(train_loader))

def sample_negatives(users,count):
    result=[]
    for u in users.tolist():
        values=[];known=all_positive[int(u)]
        while len(values)<count:
            x=random.randint(1,num_items)
            if x not in known:values.append(x)
        result.append(values)
    return torch.tensor(result,dtype=torch.long)


## 12. Ranking and auxiliary metrics

In [ ]:
class BCESASRec(nn.Module):
    def __init__(self,num_items,num_concepts,num_courses,item_concepts,item_course,item_metadata,cfg):
        super().__init__();self.cfg=cfg;d=cfg.hidden_dim
        self.item_emb=nn.Embedding(num_items+1,d,padding_idx=0)
        self.concept_emb=nn.Embedding(num_concepts+1,d,padding_idx=0)
        self.course_emb=nn.Embedding(num_courses+1,d,padding_idx=0)
        self.position_emb=nn.Embedding(cfg.max_len,d);self.time_emb=nn.Embedding(cfg.time_buckets,d,padding_idx=0)
        self.behaviour_mlp=nn.Sequential(nn.Linear(len(BEHAVIOUR_COLUMNS),64),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(64,d))
        self.metadata_mlp=nn.Sequential(nn.Linear(len(META_COLUMNS),64),nn.GELU(),nn.Linear(64,d))
        self.concept_query=nn.Linear(d,d,bias=False);self.concept_key=nn.Linear(d,d,bias=False)
        self.event_norm=nn.LayerNorm(d);self.candidate_norm=nn.LayerNorm(d);self.dropout=nn.Dropout(cfg.dropout)
        layer=nn.TransformerEncoderLayer(d,cfg.attention_heads,cfg.feedforward_dim,cfg.dropout,
                batch_first=True,norm_first=True,activation='gelu')
        self.transformer=nn.TransformerEncoder(layer,cfg.transformer_layers);self.output_norm=nn.LayerNorm(d)
        self.completion_head=nn.Sequential(nn.Linear(2*d,d),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(d,1))
        self.register_buffer('item_concepts',torch.tensor(item_concepts,dtype=torch.long))
        self.register_buffer('item_course',torch.tensor(item_course,dtype=torch.long))
        self.register_buffer('item_metadata',torch.tensor(item_metadata,dtype=torch.float32))
        self.scale=math.sqrt(d)

    def concept_pool(self,item_idx,return_weights=False):
        ids=self.item_concepts[item_idx];c=self.concept_emb(ids);q=self.concept_query(self.item_emb(item_idx)).unsqueeze(-2)
        # Calculate masking and normalization in FP32. In FP16, 1e-8 becomes zero;
        # videos with no concepts would therefore divide 0 by 0 and produce NaN.
        logits=((q*self.concept_key(c)).sum(-1)/self.scale).float();mask=ids.eq(0)
        logits=logits.masked_fill(mask,-1e9);weights=torch.softmax(logits,dim=-1)
        weights=weights.masked_fill(mask,0.0)
        weights=weights/weights.sum(-1,keepdim=True).clamp_min(1.0)
        pooled=(weights.unsqueeze(-1)*c.float()).sum(-2).to(c.dtype)
        return (pooled,weights,ids) if return_weights else pooled

    def candidate(self,item_idx):
        z=self.item_emb(item_idx)+self.concept_pool(item_idx)+self.course_emb(self.item_course[item_idx])+self.metadata_mlp(self.item_metadata[item_idx])
        return self.candidate_norm(z)

    def time_bucket(self,times):
        gap=torch.zeros_like(times);valid=(times[:,1:]>0)&(times[:,:-1]>0)
        delta=(times[:,1:]-times[:,:-1]).clamp_min(0)
        gap[:,1:]=torch.where(valid,delta,torch.zeros_like(delta))
        bucket=torch.floor(torch.log2(gap.float()+1)).long()+1
        return bucket.clamp(0,self.cfg.time_buckets-1).masked_fill(times.eq(0),0)

    def encode(self,seq,times,behaviour):
        pos=torch.arange(self.cfg.max_len,device=seq.device).unsqueeze(0)
        static=self.candidate(seq);x=static+self.behaviour_mlp(behaviour)+self.time_emb(self.time_bucket(times))+self.position_emb(pos)
        padding=seq.eq(0);x=self.dropout(self.event_norm(x));x=x.masked_fill(padding.unsqueeze(-1),0.0)
        causal=torch.triu(torch.ones(self.cfg.max_len,self.cfg.max_len,device=seq.device,dtype=torch.bool),1)
        x=self.transformer(x,mask=causal,src_key_padding_mask=padding)
        last_index=seq.ne(0).sum(1).clamp_min(1)-1
        last=x[torch.arange(len(seq),device=seq.device),last_index]
        return self.output_norm(last)

    def sampled_logits(self,h,candidates):return (h.unsqueeze(1)*self.candidate(candidates)).sum(-1)/self.scale
    def all_candidate_embeddings(self):return self.candidate(torch.arange(1,self.item_emb.num_embeddings,device=self.item_emb.weight.device))
    def completion(self,h,item):return torch.sigmoid(self.completion_head(torch.cat([h,self.candidate(item)],-1))).squeeze(-1)

model=BCESASRec(num_items,len(concept_ids),len(course_ids),item_concepts,item_course,item_metadata,CFG).to(device)
print(model)
print('Trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))


## 13. Stable training and early stopping

In [ ]:
def calculate_metrics(ranks,topk_items,item_popularity,num_items,ks=(5,10,20)):
    ranks=np.asarray(ranks,dtype=np.int64);out={'Accuracy@1':float(np.mean(ranks==1)),'MRR':float(np.mean(1/ranks)),
        'MeanRank':float(np.mean(ranks)),'MedianRank':float(np.median(ranks))}
    for k in ks:
        hit=ranks<=k;recall=float(hit.mean());precision=recall/k
        out[f'Precision@{k}']=precision;out[f'Recall@{k}']=recall
        out[f'F1@{k}']=0.0 if recall==0 else float(2*precision*recall/(precision+recall))
        out[f'NDCG@{k}']=float(np.mean(np.where(hit,1/np.log2(ranks+1),0)))
        out[f'MAP@{k}']=float(np.mean(np.where(hit,1/ranks,0)))
    rec=np.asarray(topk_items);out['CatalogCoverage@10']=float(len(np.unique(rec))/num_items)
    total=sum(item_popularity.values());probs=np.array([item_popularity.get(int(i),0.5)/total for i in rec.ravel()])
    out['Novelty@10']=float(np.mean(-np.log2(np.clip(probs,1e-12,None))))
    return out

item_popularity=train.i.value_counts().to_dict()

def tensors_for_users(histories,users):
    seq=torch.tensor([left_pad(histories[u]['items'],CFG.max_len,0) for u in users],device=device)
    times=torch.tensor([left_pad(histories[u]['times'],CFG.max_len,0) for u in users],device=device)
    beh=torch.tensor([left_pad(histories[u]['behaviour'],CFG.max_len,[0.0]*len(BEHAVIOUR_COLUMNS)) for u in users],dtype=torch.float32,device=device)
    return seq,times,beh

@torch.no_grad()
def evaluate(model,histories,targets,target_completion,users,description):
    model.eval();candidate_z=model.all_candidate_embeddings();ranks=[];top_items=[];loss_sum=0.;n=0;completion_errors=[]
    for start in tqdm(range(0,len(users),CFG.eval_batch_size),desc=description,leave=False):
        us=users[start:start+CFG.eval_batch_size];seq,times,beh=tensors_for_users(histories,us)
        h=model.encode(seq,times,beh);scores=(h@candidate_z.T)/model.scale
        if not torch.isfinite(scores).all():
            raise FloatingPointError('Non-finite evaluation scores detected; metrics were not calculated.')
        target=torch.tensor([targets[u]-1 for u in us],device=device)
        for row,u in enumerate(us):
            seen=set(histories[u]['items']);seen.discard(targets[u])
            if seen:scores[row,torch.tensor([i-1 for i in seen],device=device)]=torch.finfo(scores.dtype).min
        loss_sum+=F.cross_entropy(scores,target,reduction='sum').item();n+=len(us)
        target_scores=scores[torch.arange(len(us),device=device),target]
        ranks.extend(((scores>target_scores.unsqueeze(1)).sum(1)+1).cpu().tolist())
        top_items.extend((torch.topk(scores,k=10,dim=1).indices+1).cpu().tolist())
        completion_pred=model.completion(h,target+1).float()
        if not torch.isfinite(completion_pred).all():
            raise FloatingPointError('Non-finite completion predictions detected.')
        completion_pred=completion_pred.cpu().numpy()
        completion_true=np.asarray([target_completion[u] for u in us],dtype=np.float32)
        completion_errors.extend((completion_pred-completion_true).tolist())
    metrics=calculate_metrics(ranks,top_items,item_popularity,num_items,CFG.ks);metrics['Loss']=loss_sum/n
    err=np.asarray(completion_errors);metrics['CompletionMAE']=float(np.mean(np.abs(err)));metrics['CompletionRMSE']=float(np.sqrt(np.mean(err**2)))
    concept_recalls=[]
    for u,recs in zip(users,top_items):
        true=set(item_concepts[targets[u]])-{0};pred=set(item_concepts[np.asarray(recs)].ravel())-{0}
        if true:concept_recalls.append(len(true&pred)/len(true))
    metrics['ConceptRecall@10']=float(np.mean(concept_recalls)) if concept_recalls else float('nan')
    return metrics,ranks,top_items


## 14. Hyperparameter tuning and three-seed final evaluation

In [ ]:
train_eval_users=sorted(u for u,h in train_h.items() if len(h['items'])>=2)
train_eval_hist={};train_eval_target={};train_eval_completion={}
for u in train_eval_users:
    h=train_h[u]
    train_eval_hist[u]={k:list(v[:-1]) for k,v in h.items()}
    train_eval_target[u]=h['items'][-1]
    train_eval_completion[u]=h['completion'][-1]
print({'training_evaluation_users':len(train_eval_users),'validation_users':len(eval_users),'test_users':len(eval_users)})


## 14. Hyperparameter tuning and three-seed final evaluation

In [ ]:
def concept_pair_loss(model,h,pos_items):
    ids=model.item_concepts[pos_items];mask=ids.ne(0);has=mask.any(1)
    if not has.any():return h.sum()*0
    first=mask.float().argmax(1);positive=ids[torch.arange(len(ids),device=device),first]
    negative=torch.randint(1,model.concept_emb.num_embeddings,(len(ids),),device=device)
    negative=torch.where(negative.eq(positive),(negative%(model.concept_emb.num_embeddings-1))+1,negative)
    ps=(h*model.concept_emb(positive)).sum(-1)/model.scale;ns=(h*model.concept_emb(negative)).sum(-1)/model.scale
    return -F.logsigmoid(ps[has]-ns[has]).mean()

def fit_model(cfg,name,max_epochs,trial=None):
    seed_everything(cfg.seed)
    model=BCESASRec(num_items,len(concept_ids),len(course_ids),item_concepts,item_course,item_metadata,cfg).to(device)
    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    scaler=torch.amp.GradScaler('cuda',enabled=cfg.use_mixed_precision and device.type=='cuda')
    best=-float('inf');bad=0;history=[];path=CHECKPOINTS/f'{name}_best.pt'
    for epoch in range(1,max_epochs+1):
        started=time.time();model.train();sums=defaultdict(float);examples=0
        for users,seq,times,behaviour,pos,completion in tqdm(train_loader,desc=f'{name} {epoch:02d}/{max_epochs}',leave=False):
            users,seq,times,behaviour,pos,completion=[x.to(device,non_blocking=True) for x in [users,seq,times,behaviour,pos,completion]]
            negatives=sample_negatives(users.cpu(),cfg.negatives).to(device);candidates=torch.cat([pos[:,None],negatives],1)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type,enabled=cfg.use_mixed_precision and device.type=='cuda'):
                h=model.encode(seq,times,behaviour);logits=model.sampled_logits(h,candidates)
                ranking=F.cross_entropy(logits,torch.zeros(len(seq),dtype=torch.long,device=device),label_smoothing=.03)
                concept=concept_pair_loss(model,h,pos);pred=model.completion(h,pos)
                completion_loss=F.mse_loss(pred,completion.clamp(0,1))
                total=ranking+cfg.concept_loss_weight*concept+cfg.completion_loss_weight*completion_loss
            if not torch.isfinite(total):raise FloatingPointError(f'Non-finite loss: {name}, epoch {epoch}')
            scaler.scale(total).backward();scaler.unscale_(optimizer)
            grad=nn.utils.clip_grad_norm_(model.parameters(),cfg.gradient_clip)
            if not torch.isfinite(grad):raise FloatingPointError(f'Non-finite gradient: {name}, epoch {epoch}')
            scaler.step(optimizer);scaler.update();bs=len(seq);examples+=bs
            for k,v in [('TrainLoss',total),('RankingLoss',ranking),('ConceptLoss',concept),('CompletionLoss',completion_loss)]:sums[k]+=float(v.detach())*bs
        val,_,_=evaluate(model,valid_hist,valid_target,valid_completion,eval_users,'Validation')
        row={'Epoch':epoch,**{k:v/examples for k,v in sums.items()},**{f'Val_{k}':v for k,v in val.items()},'Seconds':time.time()-started}
        history.append(row);score=val['NDCG@10']
        print(f"{name} epoch {epoch}: loss={row['TrainLoss']:.4f}, val NDCG@10={score:.4f}, val Recall@10={val['Recall@10']:.4f}")
        if score>best+cfg.early_stopping_min_delta:
            best=score;bad=0;torch.save({'model_state':model.state_dict(),'epoch':epoch,'validation_metrics':val,'config':asdict(cfg)},path)
        else:bad+=1
        if trial is not None:
            trial.report(score,epoch)
            if trial.should_prune():raise optuna.TrialPruned()
        if epoch>=cfg.minimum_epochs and bad>=cfg.early_stopping_patience:break
    pd.DataFrame(history).to_csv(REPORTS/f'{name}_epochs.csv',index=False)
    saved=torch.load(path,map_location=device);model.load_state_dict(saved['model_state'])
    return model,saved,pd.DataFrame(history)


## 14. Hyperparameter tuning and three-seed final evaluation

In [ ]:
# ============================================================
# THREE-SEED HYPERPARAMETER EXPERIMENT


## 14. Hyperparameter tuning and three-seed final evaluation

In [ ]:
# ============================================================
try:
    import optuna
except ImportError:
    import subprocess,sys
    subprocess.check_call([sys.executable,'-m','pip','install','-q','optuna'])
    import optuna

SEEDS=[42,2026,3407]
N_TRIALS=15
TUNING_EPOCHS=8
FINAL_EPOCHS=30
TARGET_TEST_NDCG=0.90

def objective(trial):
    cfg=copy.deepcopy(CFG)
    cfg.seed=42
    cfg.hidden_dim=trial.suggest_categorical('hidden_dim',[128,256])
    cfg.transformer_layers=trial.suggest_categorical('transformer_layers',[2,3])
    cfg.attention_heads=trial.suggest_categorical('attention_heads',[4,8])
    cfg.feedforward_dim=trial.suggest_categorical('feedforward_dim',[256,512,1024])
    cfg.dropout=trial.suggest_float('dropout',0.08,0.25)
    cfg.learning_rate=trial.suggest_categorical('learning_rate',[1e-4,2e-4,3e-4,5e-4])
    cfg.weight_decay=trial.suggest_categorical('weight_decay',[1e-6,1e-5,1e-4])
    cfg.negatives=trial.suggest_categorical('negatives',[50,100,150])
    cfg.concept_loss_weight=trial.suggest_float('concept_loss_weight',0.08,0.30)
    cfg.completion_loss_weight=trial.suggest_categorical('completion_loss_weight',[0.02,0.05,0.10])
    cfg.gradient_clip=trial.suggest_categorical('gradient_clip',[0.5,1.0])
    cfg.use_mixed_precision=False
    cfg.minimum_epochs=4;cfg.early_stopping_patience=3
    model,saved,_=fit_model(cfg,f'hpo_trial_{trial.number:02d}',TUNING_EPOCHS,trial)
    score=float(saved['validation_metrics']['NDCG@10'])
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return score

study=optuna.create_study(
    study_name='bce_sasrec_validation_hpo',
    storage='sqlite:////content/drive/MyDrive/DataCon/bce_sasrec_hpo_3seeds/optuna_study.db',
    load_if_exists=True,
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42,multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=4,n_warmup_steps=4))
remaining_trials=max(0,N_TRIALS-len(study.trials))
print(f'HPO trials already stored: {len(study.trials)}; remaining: {remaining_trials}')
if remaining_trials:
    study.optimize(objective,n_trials=remaining_trials,gc_after_trial=True,
                   show_progress_bar=True,catch=(FloatingPointError,))
trials=study.trials_dataframe();trials.to_csv(REPORTS/'three_seed_hpo_trials.csv',index=False)
print('Best validation NDCG@10 during tuning:',study.best_value)
print('Best parameters:',study.best_params)

all_rows=[];histories={};manifests={}
for seed in SEEDS:
    cfg=copy.deepcopy(CFG)
    for key,value in study.best_params.items():setattr(cfg,key,value)
    cfg.seed=seed;cfg.max_epochs=FINAL_EPOCHS;cfg.minimum_epochs=10
    cfg.use_mixed_precision=False
    cfg.early_stopping_patience=6;cfg.early_stopping_min_delta=1e-4
    name=f'Tuned_BCE_SASRec_seed_{seed}'
    model,saved,history=fit_model(cfg,name,FINAL_EPOCHS)
    histories[seed]=history
    train_m,_,_=evaluate(model,train_eval_hist,train_eval_target,train_eval_completion,train_eval_users,f'Train seed {seed}')
    valid_m,_,_=evaluate(model,valid_hist,valid_target,valid_completion,eval_users,f'Validation seed {seed}')
    test_m,_,_=evaluate(model,test_hist,test_target,test_completion,eval_users,f'Test seed {seed}')
    for split,metrics in [('Train',train_m),('Validation',valid_m),('Test',test_m)]:
        all_rows.append({'Seed':seed,'Split':split,'BestEpoch':saved['epoch'],**metrics})
    manifests[str(seed)]={'best_epoch':saved['epoch'],'validation_checkpoint_metrics':saved['validation_metrics'],
                          'train_metrics':train_m,'validation_metrics':valid_m,'test_metrics':test_m}
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

results=pd.DataFrame(all_rows)
results.to_csv(REPORTS/'three_seed_all_metrics.csv',index=False)
metrics=['NDCG@10','Recall@10','MRR','Accuracy@1','ConceptRecall@10','Recall@20']
requested=results[['Seed','Split','BestEpoch']+metrics]
requested.to_csv(REPORTS/'three_seed_requested_metrics.csv',index=False)
print('\nPer-seed results');print(requested.to_string(index=False))

summary=results.groupby('Split')[metrics].agg(['mean','std']).reset_index()
summary.to_csv(REPORTS/'three_seed_mean_std.csv',index=False)
print('\nThree-seed mean ± standard deviation');print(summary.to_string(index=False))

# Graph 1: learning curves for all three seeds.
fig,axes=plt.subplots(1,2,figsize=(14,5))
for seed,h in histories.items():
    axes[0].plot(h.Epoch,h.TrainLoss,label=f'Seed {seed}')
    axes[1].plot(h.Epoch,h['Val_NDCG@10'],label=f'Seed {seed}')
axes[0].set(title='Training loss by seed',xlabel='Epoch',ylabel='Loss')
axes[1].set(title='Validation NDCG@10 by seed',xlabel='Epoch',ylabel='NDCG@10')
for ax in axes:ax.grid(alpha=.25);ax.legend()
fig.tight_layout();fig.savefig(REPORTS/'01_learning_curves.png',dpi=200,bbox_inches='tight');plt.close(fig)

# Graph 2: split comparison with seed variability.
means=results.groupby('Split')[metrics].mean().reindex(['Train','Validation','Test'])
stds=results.groupby('Split')[metrics].std().reindex(['Train','Validation','Test'])
fig,ax=plt.subplots(figsize=(14,6));means.T.plot.bar(yerr=stds.T,ax=ax,capsize=3)
ax.set(title='Train, validation and test metrics (mean ± SD, 3 seeds)',ylabel='Score',ylim=(0,1.05))
ax.grid(axis='y',alpha=.25);plt.xticks(rotation=20);fig.tight_layout()
fig.savefig(REPORTS/'02_split_metric_comparison.png',dpi=200,bbox_inches='tight');plt.close(fig)

# Graph 3: test stability across seeds.
test_rows=results[results.Split=='Test'].set_index('Seed')[metrics]
fig,ax=plt.subplots(figsize=(13,5));test_rows.plot(marker='o',ax=ax)
ax.axhline(TARGET_TEST_NDCG,color='red',linestyle='--',label='0.90 target')
ax.set(title='Test metric stability across seeds',ylabel='Score',ylim=(0,1.05));ax.grid(alpha=.25);ax.legend(ncol=4)
fig.tight_layout();fig.savefig(REPORTS/'03_test_seed_stability.png',dpi=200,bbox_inches='tight');plt.close(fig)

# Graph 4: HPO trial values.
complete=trials[trials.state=='COMPLETE'] if 'state' in trials else trials
fig,ax=plt.subplots(figsize=(10,5));ax.plot(complete.number,complete.value,marker='o')
ax.axhline(.90,color='red',linestyle='--',label='0.90 target');ax.set(title='Validation NDCG@10 across HPO trials',xlabel='Trial',ylabel='NDCG@10')
ax.grid(alpha=.25);ax.legend();fig.tight_layout();fig.savefig(REPORTS/'04_hpo_trials.png',dpi=200,bbox_inches='tight');plt.close(fig)

# Graph 5: Optuna parameter importance.
try:
    importance=optuna.importance.get_param_importances(study)
    imp=pd.Series(importance).sort_values()
    fig,ax=plt.subplots(figsize=(9,6));imp.plot.barh(ax=ax);ax.set(title='Hyperparameter importance',xlabel='Importance')
    fig.tight_layout();fig.savefig(REPORTS/'05_hyperparameter_importance.png',dpi=200,bbox_inches='tight');plt.close(fig)
except Exception as error:print('Importance graph skipped:',error)

# Animated learning curves. Pillow is used so no external video encoder is needed.
try:
    from matplotlib.animation import FuncAnimation, PillowWriter
    max_frame=max(int(h.Epoch.max()) for h in histories.values())
    fig,axes=plt.subplots(1,2,figsize=(13,5))
    colors={42:'#4C78A8',2026:'#F58518',3407:'#54A24B'}
    def animate(frame):
        for ax in axes:ax.clear()
        for seed,h in histories.items():
            visible=h[h.Epoch<=frame]
            axes[0].plot(visible.Epoch,visible.TrainLoss,label=f'Seed {seed}',color=colors[seed],marker='o',markersize=2)
            axes[1].plot(visible.Epoch,visible['Val_NDCG@10'],label=f'Seed {seed}',color=colors[seed],marker='o',markersize=2)
        axes[0].set(title=f'Training loss through epoch {frame}',xlabel='Epoch',ylabel='Loss')
        axes[1].set(title=f'Validation NDCG@10 through epoch {frame}',xlabel='Epoch',ylabel='NDCG@10',ylim=(0,1))
        for ax in axes:ax.grid(alpha=.25);ax.legend(loc='best')
        fig.tight_layout()
    animation=FuncAnimation(fig,animate,frames=range(1,max_frame+1),interval=500,repeat=True)
    animation.save(REPORTS/'05_training_animation.gif',writer=PillowWriter(fps=2),dpi=110)
    plt.close(fig)
except Exception as error:print('Training animation skipped:',error)

test_ndcg=results[results.Split=='Test']['NDCG@10']
verdict={'target':TARGET_TEST_NDCG,'test_mean':float(test_ndcg.mean()),'test_std':float(test_ndcg.std()),
         'all_seeds_above_target':bool((test_ndcg>=TARGET_TEST_NDCG).all())}
json.dump({'seeds':SEEDS,'best_hyperparameters':study.best_params,'hpo_best_validation_ndcg':study.best_value,
           'runs':manifests,'target_verdict':verdict},open(REPORTS/'three_seed_manifest.json','w'),indent=2)
print('\nTarget verdict:',verdict)
if not verdict['all_seeds_above_target']:
    print('The honest test result is below 0.90 for at least one seed; no metric was altered.')

import shutil
zip_path=shutil.make_archive('/content/drive/MyDrive/DataCon/Three_Seed_Tuned_BCE_SASRec_outputs','zip',root_dir=str(OUT.parent),base_dir=OUT.name)
print('Outputs:',REPORTS);print('ZIP:',zip_path)


## Outputs

All artifacts persist in `/content/drive/MyDrive/DataCon/bce_sasrec_hpo_3seeds`. The final downloadable archive is `/content/drive/MyDrive/DataCon/Three_Seed_Tuned_BCE_SASRec_outputs.zip`.

The main table is `reports/three_seed_requested_metrics.csv`; the complete table is `reports/three_seed_all_metrics.csv`; the animated learning curve is `reports/05_training_animation.gif`.
